# 🛠️ Notebook 05: Multi-SDK Cross-Verification (`Qiskit`, `PennyLane`, `Cirq`) & Paulihedral Compiler
**PravahaQ (`v0.1.0`) — Hexagonal SDK Parity, CNOT Staircase Cancellation & OpenQASM 3.0 / CUDA-Q**

---

## 🟢 Layer 1: Why Multi-SDK & Compiler Optimization Matter

- **Zero Vendor Lock-In:** Whether a customer runs on **IBM Quantum Heron (`Qiskit`)**, **Xanadu/AWS (`PennyLane`)**, **Google Willow (`Cirq`)**, or an **NVIDIA Grace Hopper supercomputer (`CUDA-Q`)**, PravahaQ compiles the exact same `IsingSpinGlass` tensor to every backend.
- **Paulihedral Gate Cancellation:** When two 3-body cubic gates share a 2-qubit prefix, their adjacent `CX` gates cancel ($\text{CX}_{i,j} \cdot \text{CX}_{i,j} = \hat{I}$), cutting two-qubit gate counts by **25%–45%** before touching hardware.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, hubo_to_ising
from pravaha_q.solvers import (
    ExactSpectralSolver,
    QiskitBFDCQOSolver,
    PennyLanePCESolver,
    CirqGridSolver,
)

prob = generate_coupled_problem(num_nodes=2, num_tasks=3, num_power_states=1, seed=42)
ising = hubo_to_ising(VaisheshikaHUBOCompiler().compile(prob))

results = [
    ExactSpectralSolver().solve(ising),
    QiskitBFDCQOSolver(shots=1024, seed=42).solve(ising),
    PennyLanePCESolver(num_qubits=4, seed=42).solve(ising),
    CirqGridSolver(seed=42).solve(ising),
]

print("=== Head-to-Head Multi-SDK Verification Matrix ===")
print("Adapter / Engine       | SDK Target    | Best Bitstring | Best Energy | Time (ms)")
print("-" * 81)
for r in results:
    print(f"{r.solver_name:22s} | {r.sdk_name:13s} |   |{r.best_bitstring}>   |  {r.best_energy:9.4f}  | {r.execution_time_ms:8.2f}")

=== Head-to-Head Multi-SDK Verification Matrix ===
Adapter / Engine       | SDK Target    | Best Bitstring | Best Energy | Time (ms)
---------------------------------------------------------------------------------
ExactSpectral          | NumPy/SciPy   |   |01100101>   |   -23.1480  |     0.42
BF-DCQO                | Qiskit-v2     |   |01100101>   |   -23.1480  |     8.64
Multistage-PCE (4Q)    | PennyLane     |   |01100101>   |   -23.1480  |    19.35
Cirq-GridQAOA          | Google-Cirq   |   |01100101>   |   -23.1480  |     1.12


In [2]:
from pravaha_q.compiler import compile_paulihedral_schedule, export_openqasm3, export_cudaq_kernel_source

print("=== Paulihedral Gate Cancellation & Topology Routing Comparison ===")
print("Topology Target           | Naive CX | Canceled CX | Routed 2Q Gates | CX Saved %")
print("-" * 81)
for topo in ("heavy_hex", "planar_grid", "all_to_all"):
    rep = compile_paulihedral_schedule(ising, topology=topo)
    print(
        f"{rep.topology_name:25s} |   {rep.naive_cx_count:3d}    |     {rep.paulihedral_canceled_cx:3d}     |"
        f"       {rep.total_routed_2q_gates:3d}       |   {rep.cx_reduction_percentage:5.2f}%"
    )

=== Paulihedral Gate Cancellation & Topology Routing Comparison ===
Topology Target           | Naive CX | Canceled CX | Routed 2Q Gates | CX Saved %
---------------------------------------------------------------------------------
IBM-Heron-HeavyHex-8Q     |    56    |     24      |        68       |   42.86%
Google-Willow-Grid-8Q     |    56    |     24      |        56       |   42.86%
IonQ-Forte-All2All-8Q     |    56    |     24      |        32       |   42.86%
